In [1]:
import json
import pandas as pd

with open ("json_file_from_LIMEexplainability", "r") as f:
    ex = json.load(f)

df = pd.read_csv("csv_file_with_MIMIC_data")

len(ex), len(df)

(1425, 1425)

In [2]:
from spacy.lang.es.stop_words import STOP_WORDS

/gaueko0/users/avarela/venv/lib/python3.9/site-packages/cupy/_environment.py:596: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy, cupy-cuda12x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''


In [4]:
rows = []
for filename, labels in ex.items():
    entry = {'file': filename}
    for label_name, values in labels.items():
        entry[f"{label_name}_pred"] = values.get('prediction')
        entry[f"{label_name}_conf"] = values.get('confidence')
        
        entry[f"{label_name}_thresholds"] = values.get('thresholds', {})
    rows.append(entry)

json_df = pd.DataFrame(rows)

In [6]:
df['file'] = df['file'].astype(str)
json_df['file'] = json_df['file'].astype(str)

final_df = pd.merge(df, json_df, how = "right", on = "file")

In [7]:
from sklearn.metrics import classification_report

categories = [
    "sdoh_community_present", "sdoh_community_absent", "sdoh_education", "sdoh_economics", 
    "sdoh_environment", "behavior_alcohol", "behavior_tobacco", "behavior_drug"
]

for cat in categories:

    print(f"Category {cat}")
    
    pred_col = f"{cat}_pred"

    report = classification_report(
        final_df[cat], 
        final_df[pred_col], 
        output_dict=True
    )
    print(pd.DataFrame(report))
    print("\n")

Category sdoh_community_present
                  0.0         1.0  accuracy    macro avg  weighted avg
precision    0.953757    0.985651  0.974035     0.969704      0.974281
recall       0.974409    0.973828  0.974035     0.974119      0.974035
f1-score     0.963973    0.979704  0.974035     0.971838      0.974096
support    508.000000  917.000000  0.974035  1425.000000   1425.000000


Category sdoh_community_absent
                   0.0         1.0  accuracy    macro avg  weighted avg
precision     0.991406    0.910345  0.983158     0.950876      0.983272
recall        0.989860    0.923077  0.983158     0.956468      0.983158
f1-score      0.990632    0.916667  0.983158     0.953649      0.983210
support    1282.000000  143.000000  0.983158  1425.000000   1425.000000


Category sdoh_education
                   0.0        1.0  accuracy    macro avg  weighted avg
precision     0.998555   0.780488  0.992281     0.889521      0.993352
recall        0.993530   0.941176  0.992281     0.96

In [8]:
import ast
from collections import defaultdict

def clean_ner(lista):
    hartu = defaultdict(list)

    lista = ast.literal_eval(lista)
    for instance in lista:
        if instance[1] != "O":
            hartu[instance[1][2:]].append(instance[0])

    return hartu

final_df["ner_kw"] = final_df["predicted_ner"].apply(clean_ner)

In [10]:
import string

translator = str.maketrans('', '', string.punctuation)

def clean_set(word_list):
    cleaned = set()
    for word in word_list:
        if not isinstance(word, str): continue
        c = word.strip().lower().translate(translator)
        c = ''.join([i for i in c if not i.isdigit()])
        if c and c not in STOP_WORDS:
            cleaned.add(c)
    return cleaned

In [11]:
for i in range(0,41):

    th = round(i*0.01, 2)
    evaluation_rows = []
    
    TARGET_THRESH = f"thresh_{th}" 

    print(f"-------------------------- TARGET THRESHOLD: {TARGET_THRESH} --------------------------")
    
    for ind, row in final_df.iterrows():
        ner_entities = row.get("ner_kw", {})
    
        for cat in categories:

            if "absent" in cat:
                continue
                
            pred_col = f"{cat}_pred"
            current_pred_class = row.get(pred_col)
            if "community" in cat:
                current_pred_class = 0
                if row.get(pred_col) == 1 and row.get("sdoh_community_absent_pred") == 0:
                    current_pred_class = "Present"
                elif row.get(pred_col) == 0 and row.get("sdoh_community_absent_pred") == 1:
                    current_pred_class = "Absent"
                elif row.get(pred_col) == 1 and row.get("sdoh_community_absent_pred") == 1:
                    current_pred_class = "Both"
                
            if current_pred_class in [0,4]:
                continue
    
            threshold_dict = row.get(f"{cat}_thresholds", {})
            lime_kws_raw = threshold_dict.get(TARGET_THRESH, [])
            
            if not isinstance(lime_kws_raw, list):
                lime_kws_raw = []
    
            ner_label = cat
            if "community" in ner_label: 
                ner_label = "sdoh_community"
                threshold_dict = row.get(f"sdoh_community_absent_thresholds", {})
                lime_kws_raw += threshold_dict.get(TARGET_THRESH, [])
                
            gt_raw = ner_entities.get(ner_label, [])
    
            lime_set = clean_set(lime_kws_raw)
            gt_set = clean_set(gt_raw)
    
            hits = 0
            for g_word in gt_set:
                if any(l_word == g_word for l_word in lime_set):
                    hits += 1
            
            precision = hits / len(lime_set) if len(lime_set) > 0 else 0
            recall = hits / len(gt_set) if len(gt_set) > 0 else 0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
            evaluation_rows.append({
                "category": cat,
                "prediction_class": current_pred_class,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "lime_count": len(lime_set),
                "ner_count": len(gt_set)
            })
    
    results_df = pd.DataFrame(evaluation_rows)
    
    category_report = results_df.groupby(['category', 'prediction_class']).agg({
        'precision': 'mean',
        'recall': 'mean',
        'f1': 'mean',
        'category': 'count'
    }).rename(columns={'category': 'sample_count'}).reset_index()
    
    print(f"### Performance per category and label ###")
    print(category_report.to_string(index=False))
    print("\n")

    category_report = results_df.groupby(['category']).agg({
        'precision': 'mean',
        'recall': 'mean',
        'f1': 'mean',
        'category': 'count'
    }).rename(columns={'category': 'sample_count'}).reset_index()
    
    print(f"### Performance per category ###")
    print(category_report.to_string(index=False))
    print("\n")
    print("\n")

-------------------------- TARGET THRESHOLD: thresh_0.0 --------------------------
### Performance per category and label ###
              category prediction_class  precision   recall       f1  sample_count
      behavior_alcohol                1   0.131479 0.885629 0.218643           421
      behavior_alcohol                2   0.106607 0.918254 0.185384           105
      behavior_alcohol                3   0.108407 0.938008 0.187334           492
         behavior_drug                1   0.121823 0.839744 0.207564            26
         behavior_drug                2   0.141791 0.884028 0.237742            48
         behavior_drug                3   0.119527 0.923374 0.202783           410
      behavior_tobacco                1   0.165773 0.912846 0.268348           205
      behavior_tobacco                2   0.121958 0.851636 0.206801           428
      behavior_tobacco                3   0.128604 0.939275 0.215382           446
sdoh_community_present           Absent   0.

In [12]:
import matplotlib.pyplot as plt
from collections import Counter

TARGET_THRESH = "thresh_0.11"

def get_mistake_stats_by_category(df, categories, thresh):
    mistake_results = {}

    for cat in categories:
        if "absent" in cat: continue
        
        ccat = "sdoh_community" if "community" in cat else cat
        ner_label = ccat
        
        fp_list, fn_list = [], []
        
        for _, row in df.iterrows():
            if "community" in cat:
                lime_raw = row.get("sdoh_community_present_thresholds", {}).get(thresh, []) + \
                           row.get("sdoh_community_absent_thresholds", {}).get(thresh, [])
            else:
                lime_raw = row.get(f"{cat}_thresholds", {}).get(thresh, [])

            lime_set = clean_set(lime_raw)
            gt_set = clean_set(row.get("ner_kw", {}).get(ner_label, []))

            fp_list.extend([l for l in lime_set if not any(l == g for g in gt_set)])
            fn_list.extend([g for g in gt_set if not any(g == l for l in lime_set)])

        mistake_results[ccat] = {
            'fp_counts': Counter(fp_list),
            'fn_counts': Counter(fn_list),
            'total_fp': len(fp_list),
            'total_fn': len(fn_list)
        }
    return mistake_results

mistakes = get_mistake_stats_by_category(final_df, categories, TARGET_THRESH)

In [13]:
MIN_ERROR_COUNT = 1 

for cat, data in mistakes.items():
    print(f"\n{'='*20} CATEGORY ERROR ANALYSIS: {cat} {'='*20}")

    top_fps = data['fp_counts'].most_common(15)
    top_fns = data['fn_counts'].most_common(15)

    if top_fps:
        filtered_fps = [f"{w} ({c})" for w, c in top_fps if c >= MIN_ERROR_COUNT]
        print(f"Top Hallucinations (FPs): {', '.join(filtered_fps) if filtered_fps else 'None'}")
    else:
        print("Top Hallucinations (FPs): No errors found.")
        
    if top_fns:
        filtered_fns = [f"{w} ({c})" for w, c in top_fns if c >= MIN_ERROR_COUNT]
        print(f"Top Blind Spots (FNs):    {', '.join(filtered_fns) if filtered_fns else 'None'}")
    else:
        print("Top Blind Spots (FNs): No errors found.")


==================== CATEGORY ERROR ANALYSIS: sdoh_community ====================
Top Hallucinations (FPs): historia (133), vive (76), familiar (47), murió (23), hijos (22), drogas (15), familia (13), examen (13), dental (12), madre (11), casa (6), apoyo (6), esposa (5), falleció (5), hermana (5)
Top Blind Spots (FNs):    hijos (56), esposa (51), hija (40), hijo (39), casado (31), esposo (22), hijas (17), nietos (16), ley (15), madre (13), hermana (10), familia (9), padre (8), marido (6), hermano (6)

==================== CATEGORY ERROR ANALYSIS: sdoh_education ====================
Top Hallucinations (FPs): trabaja (6), retiró (5), universidad (5), grado (4), escuela (3), educación (3), completó (3), hijos (3), jubilado (3), hija (2), hijo (2), trabajó (2), secundaria (2), colegio (2), enseña (1)
Top Blind Spots (FNs):    médico (1), phd (1), universidad (1), colegio (1), graduó (1), secundaria (1)

==================== CATEGORY ERROR ANALYSIS: sdoh_economics ====================
Top 